In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import cv2

In [2]:
BEST_MODEL_PATH = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Best_Models\fold_4_best.h5"
OPTIMIZED_THRESHOLD = 0.10

In [ ]:
def predict_infant(image_path):
    """
    Complete pipeline: Preprocessing -> Inference -> Optimized Thresholding
    """
    try:
        # --- 2. PREPROCESSING (VGG-Face Logic) ---
        # Load image
        img = tf.io.read_file(image_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [224, 224])
        
        # Convert RGB to BGR (VGG-Face requirement)
        img_bgr = img.numpy()[..., ::-1]
        
        # Mean Subtraction (VGG-Face values)
        mean = [93.5940, 104.7624, 129.1863]
        img_normalized = img_bgr - mean
        
        # Add batch dimension (1, 224, 224, 3)
        img_final = np.expand_dims(img_normalized, axis=0)

        # --- 3. MODEL INFERENCE ---
        # Load model once or pass as argument for speed
        model = load_model(BEST_MODEL_PATH, compile=False)
        
        # Get raw probability
        # Note: best_model returns [classification_output, heatmap_output]
        preds, _ = model.predict(img_final, verbose=0)
        prob = float(preds[0][0])

        # --- 4. OPTIMIZED CLASSIFICATION ---
        # Using the 0.10 threshold for ~80% Recall
        label = "Autistic" if prob > OPTIMIZED_THRESHOLD else "Non-Autistic"
        confidence = prob if label == "Autistic" else (1 - prob)

        print(f"\nResult: {label}")
        print(f"Probability Score: {prob:.4f}")
        print(f"Sensitivity Profile: Optimized (Threshold {OPTIMIZED_THRESHOLD})")
        
        return label, prob

    except Exception as e:
        print(f"Error processing image: {e}")
        return None, None